# How Deep Agents Remember Without Keeping Everything in Context

This notebook is the live-demo companion to the talk of the same name, part of the O'Reilly Agent Memory Expert Showcase. It builds one running scenario, a market intelligence agent drafting a company briefing, and uses it to show three ways a deep agent keeps working without dragging every past token along with it: **offloading**, **subagents**, and **summarization**.

Long-term memory, an agent remembering things across entirely separate conversations, is a different topic and isn't covered here. For that and everything else, see the self-paced LangChain Academy Deep Agents course.

## Setup: connect a model

**What you'll do:** install the SDKs and provide a model key.

In [ ]:
%pip install -q deepagents langgraph langchain-anthropic faker python-dotenv

In [ ]:
from dotenv import load_dotenv

# override=True so the local .env always wins, which matters if the kernel's
# process doesn't fully inherit your shell's gateway env vars
load_dotenv(override=True)

# Routed through our LangSmith gateway automatically, no manual Anthropic
# key needed, as long as ANTHROPIC_API_KEY / ANTHROPIC_BASE_URL /
# ANTHROPIC_CUSTOM_HEADERS are set (shell env or a local .env).

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("anthropic:claude-sonnet-4-5")

# To use a different provider instead, replace the line above, for example:
# model = init_chat_model("openai:gpt-4.1")

## 1: The scenario, a market intelligence agent

The agent's job: research companies and keep a running `briefing.md`. The one tool it gets, `pull_market_report`, stands in for something like a real market-intelligence API: given a company name, it comes back with a full report of recent developments, not a tidy summary. We don't have a live feed to point at from a conference room, so `pull_market_report` uses `Faker`, a real, widely used library for generating realistic fake data, to build one instead. What matters for this notebook isn't where the text comes from, it's that a single tool call can come back enormous, which is exactly what a real research or search API does too.

In [ ]:
from langchain_core.tools import tool
from faker import Faker

fake = Faker()
Faker.seed(7)  # reproducible demo runs

def _generate_market_report(company: str) -> str:
    lines = [
        f"# Market Intelligence Report: {company}",
        "",
        f"**Sector:** {fake.bs().title()}",
        f"**Headquarters:** {fake.city()}, {fake.country()}",
        "",
        "## Recent Developments",
    ]
    report = "\n".join(lines)
    # Keep adding entries until we're comfortably past deepagents' default
    # 20,000-token (~80,000-character) tool-result offload threshold.
    while len(report) < 90_000:
        entry = f"- {fake.date_this_year()}: {company} {fake.catch_phrase().lower()}. {fake.paragraph(nb_sentences=4)}"
        report += "\n" + entry
    return report

@tool
def pull_market_report(company: str) -> str:
    """Pull the full market intelligence report for a company: recent developments, hiring signals, and competitive notes. This can be a very large document."""
    return _generate_market_report(company)

_preview = _generate_market_report("Northwind Robotics")
print(f"One report is {len(_preview):,} characters, roughly {len(_preview) // 4:,} tokens.")

## 2: Offloading, oversized results never reach the model

This is on by default in every deep agent, no extra code required. `FilesystemMiddleware` watches every tool result; anything over roughly 20,000 tokens gets written to `/large_tool_results/{tool_call_id}` instead of being inlined into the conversation. The model gets a short pointer message and a head/tail preview instead, and can `read_file` back into the full report on its own if it decides it needs more.

The cell below hands the oversized tool straight to a plain agent and asks for one company's report.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(model=model, tools=[pull_market_report])

result = agent.invoke({"messages": [{"role": "user", "content":
    "Pull the market report for Northwind Robotics and tell me how many "
    "distinct developments it lists."
}]})

for m in result['messages']:
    m.pretty_print()

Look at the `ToolMessage` above: instead of the full 90,000-character report, it's a short note saying the result was too large and was saved to the filesystem, plus a head/tail preview. The model still answered the question, it just never had to hold the whole report in context to do it.

## 3: Subagents, keep the noisy work from ever reaching the main thread

Offloading cleans up after the fact, the big result still enters the graph for a moment before it's evicted. A subagent avoids the mess in the first place: it runs in its own, separate context window with its own tools, and only reports back a distilled answer. The main agent's context never sees the raw report at all.

The cell below gives `pull_market_report` only to a `market-researcher` subagent, not to the main agent. The main agent has no choice but to delegate.

In [ ]:
market_researcher = {
    "name": "market-researcher",
    "description": (
        "Use this subagent to research a company. It pulls the full market "
        "report and distills it down to a few bullet points, keeping the "
        "raw report out of the main conversation."
    ),
    "system_prompt": (
        "You are a market researcher. Use pull_market_report to fetch the "
        "requested company's report, then return the 3 most notable "
        "developments as short bullet points. Do not return the raw report."
    ),
    "tools": [pull_market_report],
}

agent = create_deep_agent(
    model=model,
    subagents=[market_researcher],
)

result = agent.invoke({"messages": [{"role": "user", "content":
    "Delegate to the market-researcher subagent to research Northwind Robotics, "
    "then add its findings as a new entry in briefing.md."
}]})

for m in result['messages']:
    m.pretty_print()

No `large_tool_results` eviction notice shows up on the main thread this time, because the raw report never reached it. The `market-researcher` subagent called `pull_market_report`, read the huge result inside its own isolated context, and handed back only the three bullet points. Same underlying tool, very different footprint on the main conversation.

## 4: Summarization, compacting the conversation once it gets long

Offloading and subagents both keep individual messages small. Summarization handles the other axis: what happens once a conversation itself has been going for a while. By default, once the conversation reaches about 85% of the model's real context window, `SummarizationMiddleware` compacts the older messages into a summary before the next model call, while the full, un-summarized history is preserved by writing it to `/conversation_history/{session_id}.md`. Nothing is lost, the model just stops paying for all of it on every turn.

A model's real context window is far too large to fill in a live demo, so, following the same trick used in the LangChain Academy course, the cell below shrinks the model's actual input and output token budgets (via `model.profile` and `model.max_tokens`) so summarization triggers after a short back-and-forth instead of the real ~200k-token threshold. The agent is also told not to use tools, just to keep the conversation's growth predictable for the demo.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Shrink the model's real context window (both the input budget it reasons
# about and the output tokens it reserves) so summarization triggers after a
# handful of turns instead of the real ~200k-token threshold.
model.profile = {**model.profile, "max_input_tokens": 5000}
model.max_tokens = 256

agent = create_deep_agent(
    model=model,
    checkpointer=InMemorySaver(),
    system_prompt=(
        "You are a market research assistant. Respond in exactly one short "
        "sentence. Never use tools; just answer directly from the conversation."
    ),
)

config = {"configurable": {"thread_id": "briefing-session"}}

In [ ]:
# 14 fictional companies for the agent to track across the conversation.
# Real, distinct names (not random Faker ones) so the final "which one did I
# mention first?" check is a meaningful test of what summarization preserved.
companies = [
    ("Northwind Robotics", "industrial automation", "2014", "400"),
    ("Bluepeak Analytics", "supply chain analytics", "2018", "120"),
    ("Solace Freight", "logistics", "2009", "900"),
    ("Alta Vista Systems", "enterprise security software", "2011", "650"),
    ("Marrow Biotech", "diagnostics hardware", "2016", "80"),
    ("Coral Reef Media", "streaming infrastructure", "2019", "220"),
    ("Harrow Metals", "industrial materials", "2005", "1200"),
    ("Pinecrest Labs", "battery research", "2020", "45"),
    ("Granite Peak Foods", "food distribution", "2008", "300"),
    ("Lumen Dynamics", "lighting technology", "2013", "150"),
    ("Verdant Grove Agritech", "precision agriculture", "2017", "95"),
    ("Ashford Maritime", "shipping and port logistics", "2003", "2000"),
    ("Cobalt Peak Mining", "resource extraction", "2001", "1800"),
    ("Fernwood Publishing", "digital media", "2012", "60"),
]

In [ ]:
turns = [f"I'm building a briefing on several companies, starting with {companies[0][0]}."]
turns += [
    f"{name} is in {sector}, founded in {year}, with about {emp} employees."
    for name, sector, year, emp in companies
]
turns.append("Which company did I mention first?")

for i, message in enumerate(turns, 1):
    result = agent.invoke({"messages": [{"role": "user", "content": message}]}, config=config)
    event = agent.get_state(config).values.get("_summarization_event")

    # Only print the turns that matter: the first one, the one where
    # summarization kicks in, and the last one with the final answer.
    if i in (1, len(turns)) or event:
        print(f"Turn {i}  User:  {message}")
        print(f"Turn {i}  Agent: {result['messages'][-1].content}")
        if event:
            cutoff = event["cutoff_index"]
            print(f"  model saw: summary + messages[{cutoff}:]  <- SUMMARIZED")
        print()

Once `_summarization_event` shows up in state, the model is no longer seeing every turn verbatim, it's seeing a summary plus whatever's happened most recently. The last question ("which one did I mention first?") is a good check: if the agent still gets it right after summarization has kicked in, the compacted history preserved what mattered.

## OPTIONAL: Tuning the thresholds

Both mechanisms above are configurable, not fixed:

- Offloading: `FilesystemMiddleware`'s `tool_token_limit_before_evict` (default 20,000) and `human_message_token_limit_before_evict` (default 50,000) control how big a result or message has to be before it's evicted. A few tools are always exempt (`ls`, `glob`, `grep`, `read_file`, `edit_file`, `write_file`), since evicting their output would defeat the point of a filesystem.
- Summarization: `SummarizationMiddleware`'s `trigger` and `keep` control when compaction fires and how much recent history it always leaves untouched.

Neither needs to be touched for the defaults to be useful; they're there for when a specific agent's workload calls for a different balance.

## Wrap-up

Three mechanisms, one goal: let a deep agent keep working on a task without paying, on every single turn, for everything it has ever seen. Offloading catches an oversized result after the fact. Subagents keep noisy work from reaching the main thread in the first place. Summarization compacts a conversation once it's been running for a while, without throwing anything away.

Long-term memory, carrying facts across entirely separate sessions, is the natural next question and isn't covered here. For that, plus backends, skills, sandboxes, and deployment, see the self-paced LangChain Academy Deep Agents course.